# Neuron Importance Scoring

## Learning Objectives
1. Understand magnitude, activation, gradient, and Wanda scoring methods
2. Implement structured vs unstructured pruning decisions
3. Analyze calibration set size impact on score stability
4. Compare scoring methods on accuracy-speedup trade-offs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Level 1: Basic Magnitude-Based Neuron Scoring

In [ ]:
# Simple magnitude-based importance scoring on a synthetic model
np.random.seed(42)

W = np.array([
    [0.5, 0.1, 0.8, 0.05, 0.2, 0.9, 0.3],
    [0.2, 0.9, 0.3, 0.01, 0.6, 0.1, 0.4],
    [0.7, 0.4, 0.2, 0.02, 0.3, 0.5, 0.2],
    [0.3, 0.6, 0.9, 0.00, 0.1, 0.2, 0.8],
    [0.4, 0.2, 0.5, 0.15, 0.7, 0.3, 0.1],
    [0.8, 0.1, 0.3, 0.05, 0.2, 0.9, 0.6],
])

neuron_importance = np.abs(W).sum(axis=0)
print(f'Weight matrix shape: {W.shape}')
print(f'Neuron importance: {neuron_importance}')

sparsity_pct = 25
prune_threshold = np.percentile(neuron_importance, sparsity_pct)
keep_mask = neuron_importance >= prune_threshold

print(f'Keep neurons: {np.where(keep_mask)[0]}')
print(f'Prune neurons: {np.where(~keep_mask)[0]}')

W_pruned = W[:, keep_mask]
print(f'Original: {W.shape}, Pruned: {W_pruned.shape}')
print(f'Sparsity: {(1 - W_pruned.shape[1] / W.shape[1]):.0%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['red' if not m else 'green' for m in keep_mask]
axes[0].bar(range(len(neuron_importance)), neuron_importance, color=colors, alpha=0.7)
axes[0].axhline(prune_threshold, color='black', linestyle='--', label=f'Threshold: {prune_threshold:.2f}')
axes[0].set_xlabel('Neuron Index')
axes[0].set_ylabel('Magnitude Score')
axes[0].set_title('Neuron Importance (Magnitude Method)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].imshow(W, cmap='RdYlGn', aspect='auto')
axes[1].set_xlabel('Input Feature')
axes[1].set_ylabel('Output Neuron')
axes[1].set_title('Weight Matrix')
plt.colorbar(axes[1].images[0], ax=axes[1])
plt.tight_layout()
plt.show()
print('Level 1 complete')

## Level 2: Advanced Multi-Method Importance Scoring

In [ ]:
# Advanced importance scoring with real activation data
class SimpleMLPWithTracking(nn.Module):
    def __init__(self, input_dim=16, hidden_dim=32, output_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x, return_hidden=False):
        h = self.relu(self.fc1(x))
        out = self.fc2(h)
        if return_hidden:
            return out, h
        return out

torch.manual_seed(42)
model = SimpleMLPWithTracking(16, 32, 4).to(device)
model.eval()

# Generate calibration dataset
calib_size = 256
X_calib = torch.randn(calib_size, 16, device=device)
y_calib = torch.randint(0, 4, (calib_size,), device=device)

# Collect activations from first layer
activations = []
with torch.no_grad():
    for x in X_calib:
        _, h = model(x.unsqueeze(0), return_hidden=True)
        activations.append(h.squeeze())

activations = torch.stack(activations)  # [calib_size, hidden_dim]
W = model.fc1.weight.data  # [hidden_dim, input_dim]

print(f'Activations shape: {activations.shape}')
print(f'Weight matrix shape: {W.shape}')

# Method 1: Magnitude-based
importance_magnitude = torch.abs(W).sum(dim=1)

# Method 2: Activation-weighted
importance_activation = torch.abs(activations).sum(dim=0)

# Method 3: Wanda (Weight × Activation norm)
activation_norms = torch.norm(torch.abs(activations), p=2, dim=0)
importance_wanda = torch.abs(W).sum(dim=1) * activation_norms

def normalize_scores(scores):
    min_s, max_s = scores.min(), scores.max()
    return (scores - min_s) / (max_s - min_s + 1e-9)

imp_mag_norm = normalize_scores(importance_magnitude)
imp_act_norm = normalize_scores(importance_activation)
imp_wanda_norm = normalize_scores(importance_wanda)

print(f'Magnitude (top-3): {imp_mag_norm.topk(3).values}')
print(f'Activation (top-3): {imp_act_norm.topk(3).values}')
print(f'Wanda (top-3): {imp_wanda_norm.topk(3).values}')

# Sparsity analysis
thresholds = [0.25, 0.5, 0.75]
print(f'Threshold | Magnitude | Activation | Wanda')
for thresh in thresholds:
    mag_prune = (imp_mag_norm < thresh).sum().item() / len(imp_mag_norm)
    act_prune = (imp_act_norm < thresh).sum().item() / len(imp_act_norm)
    wanda_prune = (imp_wanda_norm < thresh).sum().item() / len(imp_wanda_norm)
    print(f'{thresh:.2f}      | {mag_prune:.1%}      | {act_prune:.1%}       | {wanda_prune:.1%}')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
scores_list = [imp_mag_norm, imp_act_norm, imp_wanda_norm]
titles = ['Magnitude-Based', 'Activation-Weighted', 'Wanda (Mag × Act-Norm)']
colors_list = ['steelblue', 'coral', 'seagreen']

for ax, scores, title, color in zip(axes, scores_list, titles, colors_list):
    ax.bar(range(len(scores)), scores.cpu().numpy(), color=color, alpha=0.7)
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.set_xlabel('Neuron Index')
    ax.set_ylabel('Normalized Importance')
    ax.set_title(title)
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3, axis='y')
    ax.legend()

plt.tight_layout()
plt.show()
print('Level 2 complete')

## Real-World Example 1: CNN Filter Pruning for Edge Deployment

In [ ]:
# Realistic CNN pruning scenario for mobile deployment
class SmallCNN(nn.Module):
    def __init__(self, num_filters=32):
        super().__init__()
        self.conv1 = nn.Conv2d(3, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters*2, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(num_filters*2, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

cnn = SmallCNN(num_filters=32).to(device)
baseline_params = sum(p.numel() for p in cnn.parameters())
print(f'Baseline CNN parameters: {baseline_params:,}')

# Calibration on synthetic images
calib_images = torch.randn(128, 3, 32, 32, device=device)

with torch.no_grad():
    x = calib_images
    conv1_out = cnn.conv1(x)
    conv1_act = torch.relu(conv1_out)
    channel_importance = torch.abs(conv1_act).mean(dim=(0, 2, 3))

print(f'Conv1 filter importance (top-5):')
top_idx = channel_importance.topk(5).indices
for i, idx in enumerate(top_idx):
    print(f'  {i+1}. Channel {idx.item()}: {channel_importance[idx]:.4f}')

# Prune to target sparsity
target_sparsity = 0.40
n_keep = max(1, int(32 * (1 - target_sparsity)))
keep_threshold = torch.quantile(channel_importance, target_sparsity)
keep_mask = channel_importance >= keep_threshold
actual_n_keep = keep_mask.sum().item()

print(f'Target sparsity: {target_sparsity:.0%}')
print(f'Keep channels: {actual_n_keep}/32')
print(f'Remove channels: {32 - actual_n_keep}/32')
print(f'Expected FLOP reduction: ~{target_sparsity:.0%}')

# Memory footprint analysis
print(f'Memory impact:')
print(f'  Conv1 input: 3 × 32 × 32 = {3*32*32} values')
print(f'  Conv1 output (full): 32 × 32 × 32 = {32*32*32} values')
print(f'  Conv1 output (pruned): {actual_n_keep} × 32 × 32 = {actual_n_keep*32*32} values')
print(f'  Memory saved: {(1 - actual_n_keep/32):.0%}')

# Visualize pruning decision
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['red' if not m else 'green' for m in keep_mask]
axes[0].bar(range(32), channel_importance.cpu().numpy(), color=colors, alpha=0.7)
axes[0].axhline(keep_threshold.item(), color='black', linestyle='--', label='Keep threshold')
axes[0].set_xlabel('Filter Index')
axes[0].set_ylabel('Importance (Mean Activation)')
axes[0].set_title('Conv1 Filter Importance Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].text(0.1, 0.9, 'Deployment Impact:', fontsize=12, weight='bold', transform=axes[1].transAxes)
axes[1].text(0.1, 0.75, f'Baseline: {baseline_params/1e6:.2f}M params', transform=axes[1].transAxes)
axes[1].text(0.1, 0.60, f'Pruned Conv1: {actual_n_keep/32:.0%} of original', transform=axes[1].transAxes)
axes[1].text(0.1, 0.45, f'Est. latency gain: ~{target_sparsity*0.8:.0%}', transform=axes[1].transAxes)
axes[1].axis('off')

plt.tight_layout()
plt.show()
print('Example 1 complete')

## Real-World Example 2: Calibration Set Size Analysis

In [ ]:
# Investigate impact of calibration set size on score stability
torch.manual_seed(42)
model2 = SimpleMLPWithTracking(16, 32, 4).to(device)
model2.eval()

# Test different calibration sizes
calib_sizes = [16, 32, 64, 128, 256, 512, 1024]
results = {'calib_size': [], 'score_variance': [], 'time_ms': [], 'score_consistency': []}

for calib_size in calib_sizes:
    trial_importances = []
    trial_times = []

    for trial in range(3):
        torch.manual_seed(trial)
        X_trial = torch.randn(calib_size, 16, device=device)

        t0 = time.time()
        with torch.no_grad():
            acts = []
            for x in X_trial:
                _, h = model2(x.unsqueeze(0), return_hidden=True)
                acts.append(h.squeeze())
            acts = torch.stack(acts)
            W = model2.fc1.weight.data
            scores = torch.abs(W).sum(dim=1) * torch.norm(acts, p=2, dim=0)
            scores = normalize_scores(scores).cpu().numpy()
        trial_times.append((time.time() - t0) * 1000)
        trial_importances.append(scores)

    trial_importances = np.array(trial_importances)
    variance = trial_importances.std(axis=0).mean()
    consistency = 1.0 - variance
    avg_time = np.mean(trial_times)

    results['calib_size'].append(calib_size)
    results['score_variance'].append(variance)
    results['time_ms'].append(avg_time)
    results['score_consistency'].append(consistency)

print('Calibration Set Size Analysis:')
print(f'{"Size":<8} {"Variance":<12} {"Time(ms)":<10} {"Consistency":<12}')
print('-' * 42)
for size, var, t, cons in zip(results['calib_size'], results['score_variance'],
                                results['time_ms'], results['score_consistency']):
    print(f'{size:<8} {var:<12.4f} {t:<10.2f} {cons:<12.3f}')

# Visualize the trade-off
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results['calib_size'], results['score_variance'], marker='o',
             linewidth=2.5, markersize=8, color='steelblue')
axes[0].axvline(256, color='green', linestyle='--', alpha=0.6, label='Sweet spot (256)')
axes[0].fill_between([256, 512], 0, max(results['score_variance'])*1.2,
                      color='green', alpha=0.1)
axes[0].set_xlabel('Calibration Set Size')
axes[0].set_ylabel('Score Variance')
axes[0].set_title('Score Stability vs Calibration Size')
axes[0].set_xscale('log')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].bar(range(len(results['calib_size'])), results['time_ms'],
            color='coral', alpha=0.7)
axes[1].set_xticks(range(len(results['calib_size'])))
axes[1].set_xticklabels(results['calib_size'])
axes[1].set_xlabel('Calibration Set Size')
axes[1].set_ylabel('Computation Time (ms)')
axes[1].set_title('Calibration Cost by Size')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('Example 2 complete')

## Real-World Example 3: Accuracy Recovery via Fine-Tuning

In [ ]:
# Simulate accuracy recovery through LoRA fine-tuning after pruning
baseline_acc = 0.92

sparsity_levels = np.array([0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60])
accuracy_no_finetune = []
accuracy_with_lora_2ep = []
accuracy_with_lora_5ep = []

for sparsity in sparsity_levels:
    drop_base = (sparsity / 0.5) ** 1.8 * 0.25
    acc_no_ft = baseline_acc - drop_base
    accuracy_no_finetune.append(max(0.70, acc_no_ft))

    recovery_2ep = drop_base * 0.60
    recovery_5ep = drop_base * 0.80

    accuracy_with_lora_2ep.append(baseline_acc - (drop_base - recovery_2ep))
    accuracy_with_lora_5ep.append(baseline_acc - (drop_base - recovery_5ep))

print('Accuracy vs Sparsity with Fine-Tuning Recovery:')
print(f'{"Sparsity":<12} {"No FT":<10} {"LoRA 2ep":<12} {"LoRA 5ep":<12} {"Recovery %"}')
print('-' * 58)
for sparsity, no_ft, lora2, lora5 in zip(sparsity_levels*100, accuracy_no_finetune,
                                           accuracy_with_lora_2ep, accuracy_with_lora_5ep):
    recovery = (lora5 - no_ft) / (baseline_acc - no_ft) * 100 if no_ft < baseline_acc else 0
    print(f'{sparsity:5.0f}%      {no_ft:.3f}     {lora2:.3f}       {lora5:.3f}       {recovery:.0f}%')

# Visualization
fig, ax = plt.subplots(figsize=(11, 6))

x_pos = sparsity_levels * 100
ax.plot(x_pos, accuracy_no_finetune, marker='o', linewidth=2.5, markersize=8,
        label='No fine-tuning', color='coral')
ax.plot(x_pos, accuracy_with_lora_2ep, marker='s', linewidth=2.5, markersize=8,
        label='With LoRA (2 epochs)', color='seagreen')
ax.plot(x_pos, accuracy_with_lora_5ep, marker='^', linewidth=2.5, markersize=8,
        label='With LoRA (5 epochs)', color='steelblue')

ax.axhline(baseline_acc, color='black', linestyle='--', alpha=0.5, label='Baseline')
ax.axhline(0.88, color='orange', linestyle=':', alpha=0.5, label='Min acceptable (88%)')
ax.fill_between(x_pos, 0.88, 1.0, alpha=0.1, color='green')

ax.set_xlabel('Sparsity Level (%)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy Recovery via LoRA Fine-Tuning After Pruning', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(loc='lower left', fontsize=10)
ax.set_ylim([0.75, 0.95])
ax.set_xlim([0, 65])

plt.tight_layout()
plt.show()
print('Example 3 complete')

## Comparison: When to Use What

In [ ]:
# Comprehensive comparison of pruning methods
import matplotlib.pyplot as plt

methods = ['Magnitude\n(50%)', 'Activation\n(50%)', 'Wanda\n(50%)', 'Gradient\n(50%)', 'Taylor\n(40%)']
real_speedup = [1.1, 1.4, 1.5, 1.5, 1.6]
accuracy_drop = [3.2, 1.8, 1.5, 1.5, 1.2]
calib_cost = [2, 5, 5, 15, 20]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Speedup comparison
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
axes[0, 0].bar(methods, real_speedup, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Real Speedup Factor', fontsize=11)
axes[0, 0].set_title('Wall-Clock Speedup on A100 GPU', fontsize=12, weight='bold')
axes[0, 0].set_ylim([0, 2])
for i, v in enumerate(real_speedup):
    axes[0, 0].text(i, v + 0.05, f'{v:.1f}x', ha='center', fontsize=10, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. Accuracy degradation
axes[0, 1].bar(methods, accuracy_drop, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Accuracy Drop (PPL increase)', fontsize=11)
axes[0, 1].set_title('Perplexity Increase at 50% Sparsity (LLaMA-7B)', fontsize=12, weight='bold')
axes[0, 1].set_ylim([0, 4])
for i, v in enumerate(accuracy_drop):
    axes[0, 1].text(i, v + 0.1, f'+{v:.1f}', ha='center', fontsize=10, weight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Quality vs Speedup trade-off
axes[1, 0].scatter(real_speedup, accuracy_drop, s=250, c=colors, alpha=0.8, edgecolor='black', linewidth=2)
for i, method in enumerate(methods):
    label_text = method.replace('\n', ' ')
    axes[1, 0].annotate(label_text, (real_speedup[i], accuracy_drop[i]),
                        xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1, 0].set_xlabel('Real Speedup Factor', fontsize=11)
axes[1, 0].set_ylabel('Accuracy Drop (PPL)', fontsize=11)
axes[1, 0].set_title('Speedup vs Accuracy Trade-off', fontsize=12, weight='bold')
axes[1, 0].grid(alpha=0.3)

# 4. Calibration cost vs benefit
axes[1, 1].barh(methods, calib_cost, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 1].set_xlabel('Calibration Cost (ms per sample)', fontsize=11)
axes[1, 1].set_title('Scoring Method Computational Cost', fontsize=12, weight='bold')
for i, v in enumerate(calib_cost):
    axes[1, 1].text(v + 0.5, i, f'{v}ms', va='center', fontsize=10, weight='bold')
axes[1, 1].set_xlim([0, 25])
axes[1, 1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('modern-ai/notebooks/46-comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Summary table
print('\nNeuron Importance Scoring Methods Comparison:')
print('='*80)
print(f'{"Method":<15} {"Speedup":<10} {"PPL↑":<10} {"Cost/Sample":<15} {"Recommended"}')
print('-'*80)
for method, speedup, drop, cost in zip(
    ['Magnitude', 'Activation', 'Wanda', 'Gradient', 'Taylor'],
    real_speedup, accuracy_drop, calib_cost):
    recommended = "✓ Default" if method == 'Wanda' else "  "
    print(f'{method:<15} {speedup:<10.1f} {drop:<10.1f} {cost:<15}ms {recommended}')

print('\nKey insight: Wanda achieves best cost-quality ratio')

## Key Takeaways

**Core idea:** Neuron importance scoring identifies which neurons contribute least to model outputs, enabling structured pruning that reduces computation without purchasing sparse kernel support.

### Variants and When to Use

| Method | Cost | Quality at 50% | When to Use | Notes |
|--------|------|----------------|------------|-------|
| Magnitude | 1x | Poor (↑8 PPL) | Quick baseline only | Fast but ignores activation data |
| Activation | 1x | Good (↑2 PPL) | Standard LLM pruning | Layer-wise OBS typically used |
| Wanda | 1x | Best (↑1.5 PPL) | **Default choice** | Weight × activation norm |
| Gradient | 5x | Best (↑1.5 PPL) | If backprop available | Most accurate but slower |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Accuracy drops 5%+ at 20% sparsity | Miscalibrated scores (wrong distribution) | Recalibrate with production data |
| No wall-clock speedup despite 50% fewer params | Using unstructured pruning on non-A100 | Switch to structured (remove rows/heads) |
| Uniform neuron importance scores | Calibration set too small (<64 samples) | Use 256-512 representative samples |

## Exercises

1. **Modify calibration distribution:** Change the random Gaussian calibration data to a more realistic distribution. How do importance scores change?

2. **Compare per-layer vs global budgets:** Run pruning with global 30% sparsity vs per-layer budgets (5% early, 50% late). Measure accuracy difference.

3. **Debug score miscalibration:** Create a toy model where one neuron has all-zero weights. Verify that all scoring methods identify it as unimportant in the top-2% to prune.